# Optimal Model — HuffPost News Classification

**Strategy:** RoBERTa-base two-phase fine-tuning with:
1. **Label consolidation** 41 → 35 classes (merges near-synonymous categories with F1 < 0.45)
2. **CLS token pooling** (replaces GlobalAveragePooling — CLS is what RoBERTa is trained to use for classification)
3. **Focal loss** (replaces cross-entropy + class weights — focuses training on hard/minority examples)
4. **val_accuracy early stopping** (not val_loss — these diverge on transformer fine-tuning)
5. **Warmup + cosine decay LR schedule** in Phase 2 (replaces flat 2e-5)
6. **Longer Phase 2** (up to 10 epochs with patience=3)

**Baseline to beat:** OPTIMAL v1 DistilBERT — test acc 0.6457 / macro-F1 0.5824

## Background: Milestone 1 — EDA & Problem Framing

### Dataset
The HuffPost News Category dataset contains **200,853 news articles** across **41 categories** with fields: `headline`, `short_description`, `category`, `authors`, `link`, `date`. The team selected this dataset over image classification alternatives due to its real-world NLP applicability.

### Key EDA Findings

**Class imbalance (32× ratio):**
| Category | Count |
|---|---|
| POLITICS (largest) | 32,739 |
| WELLNESS | 17,827 |
| ENTERTAINMENT | 16,058 |
| EDUCATION (smallest) | 1,004 |

**Text length statistics:**
- Headline: mean 9.5 words (std 3.1), range 0–44 words
- Description: mean 19.7 words (std 14.4), range 0–243 words
- Combined (p50 / p95 / p99): 29 / 56 / 68 words → max_length=128 truncates only ~0.1% of samples

**Data quality issues:**
- 6 empty headlines (dropped)
- 19,712 articles (~9.8%) missing `short_description` → used headline-only text
- 1,398 duplicate (headline, category) pairs
- 717 headlines appearing under multiple categories (genuine label ambiguity)

**Category overlap identified (root cause of model confusion):**
```
ARTS & CULTURE  ↔  ARTS  ↔  CULTURE & ARTS
STYLE           ↔  STYLE & BEAUTY
HEALTHY LIVING  ↔  WELLNESS
THE WORLDPOST   ↔  WORLDPOST
PARENTS         ↔  PARENTING
TASTE           ↔  FOOD & DRINK
```

### Preprocessing Pipeline

```python
# 1. Text construction
text = headline.lower() + " [sep] " + short_description.lower()

# 2. Drop near-empty samples
df = df[df['text'].str.len() > 5]

# 3. Stratified 80/10/10 split
train_test_split(..., stratify=labels)

# 4. Class weights (balanced) to address 32× imbalance
compute_class_weight("balanced", classes=np.arange(41), y=train_labels)
# Weight range: 0.150× (POLITICS) – 4.874× (EDUCATION)

# 5. Tokenization
TextVectorization(max_tokens=20000, output_sequence_length=128)
```

### Feature Engineering Decisions

- **Vocabulary cap at 20,000 tokens** — sufficient for news domain; larger vocab didn't improve results
- **Sequence length 128** — preserves 99%+ of samples; no meaningful truncation loss
- **`[sep]` separator token** — concatenates headline and description into single input sequence
- **t-SNE / PCA visualizations** — confirmed no linear separation between categories; overlap validated the need for deep architectures
- **GloVe 100d identified as promising** — EDA showed word frequency alone insufficient to distinguish overlapping categories; pretrained semantic vectors expected to help

### Primary Success Metric: Macro-F1

Accuracy is misleading on a 32× imbalanced 41-class dataset. Macro-F1 treats all classes equally — a model that ignores minority classes (e.g. EDUCATION, GOOD NEWS) is penalized regardless of overall accuracy.


## Background: Milestone 2 — Model Experiments & Key Learnings

### Experiment Summary

Three models were trained on the 41-class dataset. Results in this notebook directly informed the optimal model design.

| Model | Architecture | Test Acc | Macro-F1 | Train Time | Key Decision |
|---|---|---|---|---|---|
| Baseline | Embedding(20k,64) → GAP → Dense(64) → Dense(41) | 0.5832 | — | 4.75 min | Bag-of-words ceiling |
| Custom (BiLSTM) | GloVe(100d, trainable) → BiLSTM(64) → Dense(64) → Dense(41) | 0.6433 | — | 121.7 min | **Pretrained embeddings decisive** |
| DistilBERT | Frozen base → GAP → Dropout(0.3) → Dense(41) (two-phase in TCB) | 0.6457 | 0.5824 | 135.2 min | **Full fine-tuning essential** |

### What Each Experiment Revealed

**Baseline (Embedding + GlobalAveragePooling):**
```python
Embedding(20000, 64) → GlobalAveragePooling1D → Dense(64, relu) → Dense(41, softmax)
# Result: test_acc=0.5832, best_epoch=14, train_time=4.75 min
```
- GAP discards word order — treats each document as a bag of tokens
- Overfitting after epoch 14; early stopping at epoch 19
- Establishes the ceiling for order-insensitive models: ~58% accuracy
- **Learning:** Ordering matters for distinguishing overlapping news categories

**Custom Model (GloVe + BiLSTM):**
```python
Embedding(20000, 100, weights=[glove_matrix], trainable=True)
  → Bidirectional(LSTM(64)) → Dropout(0.3) → Dense(64, relu) → Dropout(0.2) → Dense(41, softmax)
# Result: test_acc=0.6433, best_epoch=4, train_time=121.7 min
```
Three variants were tested before settling on the best:

| Variant | Setup | Test Acc |
|---|---|---|
| V1 | GloVe frozen → GAP → Dense(64) | 0.5289 |
| V2 | GloVe frozen → Dropout(0.4) → GAP → Dense(64) | 0.4550 |
| **V3 (best)** | **GloVe trainable → BiLSTM(64) → Dropout(0.3) → Dense(64) → Dense(41)** | **0.6433** |

- Pretrained GloVe embeddings gave +6.2% over baseline — the single highest-leverage decision at this tier
- **Trainable embeddings essential:** frozen embeddings (V1) underperformed; V2 showed aggressive dropout (0.4) on frozen embeddings actively hurts
- **BiLSTM captures word order** — critical for distinguishing `POLITICS` from `WORLD NEWS`, `WELLNESS` from `HEALTHY LIVING`
- Converged at epoch 4 (fast); val_loss and val_accuracy largely agreed at this model tier

**DistilBERT Fine-tuning:**

The TCB notebook used two-phase fine-tuning (decisive advantage):
```python
# Phase 1: frozen base, train head only
distilbert_base.trainable = False
model.compile(optimizer=Adam(1e-3), ...)   # head: GAP → Dropout(0.3) → Dense(41)
# Phase 2: full fine-tune at controlled LR
distilbert_base.trainable = True
model.compile(optimizer=Adam(2e-5), ...)
# Result: test_acc=0.6457, macro-F1=0.5824, train_time=135.2 min
```

RTF notebook used frozen base only → test_acc=0.6017 (–4.4%). DistilBERT still improving at epoch 10 when stopped — not converged.

**Critical observation — val_loss vs val_accuracy divergence:**
- On the 41-class imbalanced dataset, `val_loss` bottomed out at epoch 2 for the BiLSTM
- `val_accuracy` kept improving until epoch 6
- Monitoring `val_loss` for early stopping prematurely stops training
- **Fix in optimal model: monitor `val_accuracy`, not `val_loss`**

### Per-Class F1 (DistilBERT, 41 classes)

Hardest categories (F1 < 0.40):
```
GOOD NEWS   0.352    IMPACT      0.371    EDUCATION   0.383
```
Easiest categories (F1 > 0.85):
```
STYLE & BEAUTY  0.868    WEDDINGS  0.859    SPORTS  0.844
```

Hardest categories are either minority classes or have ambiguous/overlapping content. This directly motivated the label consolidation (41 → 35) in this notebook.

### Three Improvements Implemented in This Notebook

1. **Label consolidation 41 → 35** — merges the 7 near-synonymous category pairs identified in EDA; removes systematic confusion at the data level
2. **`val_accuracy` early stopping** — replaces `val_loss` monitoring to avoid premature convergence
3. **Warmup + cosine decay LR schedule (Phase 2)** — replaces flat `lr=2e-5`; linear warmup prevents large early gradient updates on the unfrozen base, cosine decay provides smooth annealing


In [1]:
# ── Package setup ─────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["tf_keras", "transformers==4.47.1"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

import os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers

from datasets import load_dataset, DatasetDict, load_from_disk
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, classification_report
from transformers import RobertaTokenizerFast, TFRobertaModel

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# ── Mixed precision (fp16 on Ampere/RTX 3060 — ~1.7x faster, same accuracy) ──
# Must use tf_keras.mixed_precision — tf.keras and tf_keras are separate namespaces.
keras.mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", keras.mixed_precision.global_policy())

# ── Hyperparameters ───────────────────────────────────────────────────────────
SEED         = 42
BERT_MAX_LEN = 128
BERT_BATCH   = 32   # default batch (roberta-base)
LARGE_BATCH  = 8    # roberta-large Phase 2 OOMs at 16 — 8 fits within 12GB VRAM

# ── Experiment config ─────────────────────────────────────────────────────────
# Each entry: (model_name, focal_loss_gamma)
EXPERIMENTS = [
    ("roberta-base",  2.0),
    ("roberta-base",  1.5),
    ("roberta-large", 2.0),
]
PHASE1_EPOCHS_MAX  = 10
PHASE2_EPOCHS_MAX  = 40
PHASE1_PATIENCE    = 4
PHASE2_PATIENCE    = 6
PHASE2_PEAK_LR     = 2e-5
PHASE2_WARMUP_EPS  = 2           # warmup epochs before cosine decay starts

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

I0000 00:00:1776040157.072045  345870 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 3060, compute capability 8.6
Mixed precision policy: <Policy "mixed_float16">
TF version: 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Data Preparation

In [2]:
# ── Load raw dataset ──────────────────────────────────────────────────────────
DISK_PATH = "huffpost_splits"
RAW_PATH  = "huffpost.json"
URL       = "https://huggingface.co/datasets/khalidalt/HuffPost/resolve/main/News_Category_Dataset_v2.json"

if os.path.exists(DISK_PATH):
    huff = load_from_disk(DISK_PATH)
    print("Loaded from disk cache.")
elif os.path.exists(RAW_PATH):
    huff = load_dataset("json", data_files=RAW_PATH, split="train")
    print("Loaded from local JSON.")
else:
    huff = load_dataset("json", data_files=URL, split="train")
    huff.save_to_disk(DISK_PATH)
    print("Downloaded and cached to disk.")

print(f"Raw records: {len(huff):,}")

# ── Label consolidation: 41 → 35 classes ──────────────────────────────────────
# Merges categories that are near-synonymous and caused systematic model confusion.
# Evidence: DistilBERT F1 < 0.45 on all categories within each merge group.
MERGE_MAP = {
    "ARTS & CULTURE": "ARTS",           # F1 overlap: ARTS ↔ ARTS & CULTURE ↔ CULTURE & ARTS
    "CULTURE & ARTS": "ARTS",
    "STYLE":          "STYLE & BEAUTY", # Same category, renamed at some point in dataset
    "HEALTHY LIVING": "WELLNESS",       # Near-identical content and vocabulary
    "THE WORLDPOST":  "WORLDPOST",      # Same publication, different label
    "PARENTS":        "PARENTING",      # Identical audience and content
    "TASTE":          "FOOD & DRINK",   # Taste is the food/drink section of HuffPost
}

def consolidate_label(ex):
    return {"category": MERGE_MAP.get(ex["category"], ex["category"])}

huff2 = huff.map(consolidate_label)

# ── Clean and normalize text ──────────────────────────────────────────────────
def build_text(ex):
    h = (ex.get("headline") or "").strip().lower()
    s = (ex.get("short_description") or "").strip().lower()
    return {"text": (h + " [sep] " + s).strip()}

huff2 = huff2.map(build_text)
before = len(huff2)
huff2  = huff2.filter(lambda ex: len(ex["text"].strip()) > 5)
print(f"Dropped {before - len(huff2)} near-empty samples; {len(huff2):,} remaining.")

# ── Integer-encode labels ─────────────────────────────────────────────────────
huff2       = huff2.class_encode_column("category")
label_names = huff2.features["category"].names
NUM_CLASSES = len(label_names)

print(f"\nClasses after consolidation: {NUM_CLASSES}  (was 41)")
print("Merged categories:")
for src, tgt in MERGE_MAP.items():
    print(f"  {src}  →  {tgt}")

Loaded from disk cache.
Raw records: 200,853
Dropped 5 near-empty samples; 200,848 remaining.

Classes after consolidation: 34  (was 41)
Merged categories:
  ARTS & CULTURE  →  ARTS
  CULTURE & ARTS  →  ARTS
  STYLE  →  STYLE & BEAUTY
  HEALTHY LIVING  →  WELLNESS
  THE WORLDPOST  →  WORLDPOST
  PARENTS  →  PARENTING
  TASTE  →  FOOD & DRINK


In [3]:
# ── Stratified 80 / 10 / 10 split ────────────────────────────────────────────
tmp       = huff2.train_test_split(test_size=0.10, seed=SEED, stratify_by_column="category")
train_val = tmp["train"].train_test_split(test_size=1/9, seed=SEED, stratify_by_column="category")
ds = DatasetDict(train=train_val["train"], val=train_val["test"], test=tmp["test"])

print("Split sizes:")
for split, d in ds.items():
    print(f"  {split:6s}: {len(d):>7,} samples")

train_texts  = np.array(ds["train"]["text"])
train_labels = np.array(ds["train"]["category"])
val_texts    = np.array(ds["val"]["text"])
val_labels   = np.array(ds["val"]["category"])
test_texts   = np.array(ds["test"]["text"])
test_labels  = np.array(ds["test"]["category"])

# ── Class weights ─────────────────────────────────────────────────────────────
cw_array     = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_labels)
class_weight = dict(enumerate(cw_array))
print(f"\nClass weight range: {cw_array.min():.3f} – {cw_array.max():.3f}")
print(f"  Highest (minority): {label_names[int(np.argmax(cw_array))]}  ({cw_array.max():.2f}×)")
print(f"  Lowest  (majority): {label_names[int(np.argmin(cw_array))]}  ({cw_array.min():.2f}×)")

Split sizes:
  train : 160,678 samples
  val   :  20,085 samples
  test  :  20,085 samples

Class weight range: 0.180 – 5.878
  Highest (minority): EDUCATION  (5.88×)
  Lowest  (majority): POLITICS  (0.18×)


## 2. RoBERTa Tokenization

In [4]:
# ── Tokenize all splits with RoBERTa's BPE tokenizer ────────────────────────
print("Loading tokenizer...")
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

def bert_tokenize(texts, max_len=BERT_MAX_LEN):
    enc = tokenizer(
        list(texts),
        max_length=max_len,
        truncation=True,
        padding="max_length",
        return_tensors="np",
    )
    return enc["input_ids"], enc["attention_mask"]

print("Tokenizing splits (≈10s on GPU machine)...")
t0 = time.time()
tr_ids,  tr_mask  = bert_tokenize(train_texts)
val_ids, val_mask = bert_tokenize(val_texts)
te_ids,  te_mask  = bert_tokenize(test_texts)
print(f"Done in {time.time()-t0:.1f}s  |  input_ids shape: {tr_ids.shape}")

# ── tf.data pipelines ─────────────────────────────────────────────────────────
# batch_size is parameterized — roberta-large uses LARGE_BATCH=16 to fit in VRAM.
def make_bert_ds(ids, mask, labels, shuffle=False, batch_size=BERT_BATCH):
    d = tf.data.Dataset.from_tensor_slices(
        ({"input_ids": ids, "attention_mask": mask}, labels)
    )
    if shuffle:
        d = d.shuffle(len(labels), seed=SEED)
    return d.batch(batch_size).prefetch(tf.data.AUTOTUNE)

print("Pipelines ready.")


Loading tokenizer...
Tokenizing splits (≈10s on GPU machine)...
Done in 8.6s  |  input_ids shape: (160678, 128)
Pipelines ready.


## 3. Model Architecture

In [5]:
# ── Focal loss ────────────────────────────────────────────────────────────────
def focal_loss(gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0)
        probs  = tf.reduce_sum(y_pred * tf.one_hot(y_true, tf.shape(y_pred)[-1]), axis=-1)
        ce     = -tf.math.log(probs)
        weight = tf.pow(1.0 - probs, gamma)
        return tf.reduce_mean(weight * ce)
    return loss_fn

# ── Model builder ─────────────────────────────────────────────────────────────
# Reloads fresh pretrained weights each call — fully independent experiments.
# training=False omitted: Keras propagates the runtime training flag through
# the graph, enabling RoBERTa's internal dropout during model.fit().
def build_model(model_name, gamma):
    print(f"  Loading {model_name} for gamma={gamma}...")
    base = TFRobertaModel.from_pretrained(model_name)
    base.trainable = False  # frozen in Phase 1

    inp_ids  = keras.Input(shape=(BERT_MAX_LEN,), dtype=tf.int32, name="input_ids")
    inp_mask = keras.Input(shape=(BERT_MAX_LEN,), dtype=tf.int32, name="attention_mask")

    hidden  = base(inp_ids, attention_mask=inp_mask).last_hidden_state
    cls_out = hidden[:, 0, :]
    dropped = layers.Dropout(0.3)(cls_out)
    # dtype='float32' keeps final softmax in fp32 for numerical stability
    output  = layers.Dense(NUM_CLASSES, activation="softmax",
                           name="output", dtype="float32")(dropped)

    tag   = model_name.replace("roberta-", "")  # "base" or "large"
    model = keras.Model(inputs=[inp_ids, inp_mask], outputs=output,
                        name=f"roberta_{tag}_g{gamma}")
    return model, base

print("Model builder ready.")


Model builder ready.


## 4. Training

### Phase 1 — Head only (frozen base)
Train only the CLS → Dropout → Dense(35) head at a high learning rate.
Warms up the head weights before the base is unlocked.

In [ ]:
# ── Warmup + cosine decay LR schedule ────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, peak_lr, warmup_steps, total_steps):
        self.peak_lr      = float(peak_lr)
        self.warmup_steps = float(warmup_steps)
        self.total_steps  = float(total_steps)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step / self.warmup_steps)
        cos_decay = 0.5 * self.peak_lr * (
            1.0 + tf.cos(np.pi * (step - self.warmup_steps) /
                         (self.total_steps - self.warmup_steps))
        )
        return tf.where(step < self.warmup_steps, warmup_lr, cos_decay)

    def get_config(self):
        return {"peak_lr": self.peak_lr,
                "warmup_steps": self.warmup_steps,
                "total_steps":  self.total_steps}


# ── Experiment runner ─────────────────────────────────────────────────────────
all_results = {}  # keyed by (model_name, gamma)

def run_experiment(model_name, gamma, skip_phase1=False):
    """Run Phase 1 + Phase 2 for a given model and focal loss gamma.

    skip_phase1=True: load saved Phase 1 checkpoint and go straight to Phase 2.
    Use this to resume after an OOM crash that completed Phase 1 successfully.
    """
    batch_size      = LARGE_BATCH if "large" in model_name else BERT_BATCH
    steps_per_epoch = len(train_labels) // batch_size
    exp_key         = (model_name, gamma)
    tag             = model_name.replace("roberta-", "")
    out_dir         = f"results_{tag}_gamma{gamma}"

    print(f"\n{'='*64}")
    print(f"  EXPERIMENT  {model_name}  focal_loss gamma={gamma}  batch={batch_size}")
    if skip_phase1:
        print(f"  Resuming from saved Phase 1 checkpoint.")
    print(f"{'='*64}")

    os.makedirs(out_dir, exist_ok=True)

    # Rebuild pipelines with the correct batch size for this model
    train_ds = make_bert_ds(tr_ids,  tr_mask,  train_labels, shuffle=True, batch_size=batch_size)
    val_ds   = make_bert_ds(val_ids, val_mask, val_labels,   batch_size=batch_size)
    test_ds  = make_bert_ds(te_ids,  te_mask,  test_labels,  batch_size=batch_size)

    model, base = build_model(model_name, gamma)

    # ── Phase 1: frozen base, head only ───────────────────────────────────────
    if skip_phase1:
        # Load weights BEFORE any compile() call — compiling first creates a
        # LossScaleOptimizerV3 whose state load_weights() then tries to restore,
        # hitting a tf_keras bug where that wrapper has no 'name' attribute.
        p1_ckpt = f"{out_dir}/best_p1.weights.h5"
        print(f"\nLoading Phase 1 weights from {p1_ckpt}...")
        model.load_weights(p1_ckpt)
        print("Phase 1 weights loaded.")
        hist_p1 = None
        t_p1    = 0.0
    else:
        model.compile(
            optimizer=keras.optimizers.Adam(1e-3),
            loss=focal_loss(gamma=gamma),
            metrics=["accuracy"],
        )
        cb_p1 = [
            keras.callbacks.EarlyStopping(
                monitor="val_accuracy", patience=PHASE1_PATIENCE,
                restore_best_weights=True, verbose=1),
            keras.callbacks.ModelCheckpoint(
                filepath=f"{out_dir}/best_p1.weights.h5",
                monitor="val_accuracy", save_best_only=True,
                save_weights_only=True, verbose=0),
        ]
        print(f"\n── Phase 1: frozen base  "
              f"(max {PHASE1_EPOCHS_MAX} ep, patience={PHASE1_PATIENCE}) ─")
        t0 = time.time()
        hist_p1 = model.fit(train_ds, validation_data=val_ds,
                            epochs=PHASE1_EPOCHS_MAX, callbacks=cb_p1, verbose=1)
        t_p1 = time.time() - t0
        print(f"Phase 1 done — {t_p1/60:.1f} min  "
              f"({len(hist_p1.history['accuracy'])} epochs)")

    # ── Phase 2: full fine-tune, warmup + cosine ───────────────────────────────
    base.trainable = True
    total_steps  = PHASE2_EPOCHS_MAX * steps_per_epoch
    warmup_steps = PHASE2_WARMUP_EPS  * steps_per_epoch

    lr_schedule = WarmupCosineDecay(
        peak_lr=PHASE2_PEAK_LR,
        warmup_steps=warmup_steps,
        total_steps=total_steps,
    )
    model.compile(
        optimizer=keras.optimizers.Adam(lr_schedule),
        loss=focal_loss(gamma=gamma),
        metrics=["accuracy"],
    )
    cb_p2 = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", patience=PHASE2_PATIENCE,
            restore_best_weights=True, verbose=1),
        keras.callbacks.ModelCheckpoint(
            filepath=f"{out_dir}/best_p2.weights.h5",
            monitor="val_accuracy", save_best_only=True,
            save_weights_only=True, verbose=0),
    ]
    print(f"\n── Phase 2: full fine-tune  "
          f"(max {PHASE2_EPOCHS_MAX} ep, patience={PHASE2_PATIENCE}) ─")
    print(f"   Peak LR={PHASE2_PEAK_LR}, "
          f"warmup={PHASE2_WARMUP_EPS} ep ({warmup_steps:,} steps), "
          f"cosine decay over {PHASE2_EPOCHS_MAX - PHASE2_WARMUP_EPS} ep")
    t0 = time.time()
    hist_p2 = model.fit(train_ds, validation_data=val_ds,
                        epochs=PHASE2_EPOCHS_MAX, callbacks=cb_p2, verbose=1)
    t_p2 = time.time() - t0
    print(f"Phase 2 done — {t_p2/60:.1f} min  "
          f"({len(hist_p2.history['accuracy'])} epochs)")
    if not skip_phase1:
        print(f"Total training time: {(t_p1 + t_p2)/60:.1f} min")

    # ── Evaluate ───────────────────────────────────────────────────────────────
    _, val_acc  = model.evaluate(val_ds,  verbose=0)
    _, test_acc = model.evaluate(test_ds, verbose=0)
    val_preds   = np.argmax(model.predict(val_ds,  verbose=0), axis=1)
    test_preds  = np.argmax(model.predict(test_ds, verbose=0), axis=1)
    val_f1  = f1_score(val_labels,  val_preds,  average="macro")
    test_f1 = f1_score(test_labels, test_preds, average="macro")

    print(f"\n── Results ───────────────────────────────────────────────────")
    print(f"  Val  accuracy={val_acc:.4f}  Val  macro-F1={val_f1:.4f}")
    print(f"  Test accuracy={test_acc:.4f}  Test macro-F1={test_f1:.4f}")

    # ── Persist ───────────────────────────────────────────────────────────────
    with open(f"{out_dir}/metrics.txt", "w") as fh:
        fh.write(f"model={model_name}\ngamma={gamma}\nbatch={batch_size}\n"
                 f"val_accuracy={val_acc:.4f}\nval_f1={val_f1:.4f}\n"
                 f"test_accuracy={test_acc:.4f}\ntest_f1={test_f1:.4f}\n"
                 f"phase1_epochs={'skipped' if skip_phase1 else len(hist_p1.history['accuracy'])}\n"
                 f"phase2_epochs={len(hist_p2.history['accuracy'])}\n"
                 f"train_time_min={(t_p1 + t_p2)/60:.1f}\n")

    with open(f"{out_dir}/classification_report.txt", "w") as fh:
        fh.write(classification_report(
            test_labels, test_preds, target_names=label_names, zero_division=0))

    all_results[exp_key] = dict(
        model_name=model_name, gamma=gamma, batch_size=batch_size,
        hist_p1=hist_p1, hist_p2=hist_p2,
        val_acc=val_acc,   val_f1=val_f1,
        test_acc=test_acc, test_f1=test_f1,
        test_preds=test_preds, val_preds=val_preds,
        t_p1=t_p1, t_p2=t_p2, out_dir=out_dir,
    )
    return all_results[exp_key]

print(f"Experiment runner ready.")

### Phase 2 — Full fine-tune (warmup + cosine decay)
Unfreeze all 125M RoBERTa parameters. Linear warmup over 1 epoch brings the LR from 0 → 2e-5,
then cosine decay reduces it smoothly to ~0 over the remaining epochs.
This prevents catastrophic forgetting while allowing full domain adaptation.

In [ ]:
# ── Run all experiments sequentially ─────────────────────────────────────────
for model_name, gamma in EXPERIMENTS:
    # roberta-base γ=2.0 already completed successfully — skip to avoid re-running.
    if model_name == "roberta-base" and gamma == 2.0:
        print(f"Skipping {model_name} γ={gamma} — already completed.")
        continue
    # roberta-large Phase 1 already completed before the OOM crash.
    # Load the saved checkpoint and skip straight to Phase 2 (batch=8 fixes OOM).
    skip = (model_name == "roberta-large")
    run_experiment(model_name, gamma, skip_phase1=skip)

print("\n\nAll experiments complete.")

Skipping roberta-base γ=2.0 — already completed.

  EXPERIMENT  roberta-base  focal_loss gamma=1.5  batch=32
  Loading roberta-base for gamma=1.5...


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaModel: ['lm_head.bias', 'roberta.embeddings.position_ids', 'lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight']
- This IS expected if you are initializing TFRobertaModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaModel were not initialized from the PyTorch model and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and infe


── Phase 1: frozen base  (max 10 ep, patience=4) ─
Epoch 1/10
5022/5022 [==============================] - 746s 147ms/step - loss: 2.2132 - accuracy: 0.3736 - val_loss: 1.7215 - val_accuracy: 0.4914
Epoch 2/10
5022/5022 [==============================] - 741s 148ms/step - loss: 1.7791 - accuracy: 0.4742 - val_loss: 1.4465 - val_accuracy: 0.5538
Epoch 3/10
5022/5022 [==============================] - 737s 147ms/step - loss: 1.6644 - accuracy: 0.5005 - val_loss: 1.3197 - val_accuracy: 0.5705
Epoch 4/10
5022/5022 [==============================] - 737s 147ms/step - loss: 1.6156 - accuracy: 0.5147 - val_loss: 1.2440 - val_accuracy: 0.5873
Epoch 5/10
5022/5022 [==============================] - 738s 147ms/step - loss: 1.5922 - accuracy: 0.5210 - val_loss: 1.2046 - val_accuracy: 0.5948
Epoch 6/10
5022/5022 [==============================] - 737s 147ms/step - loss: 1.5763 - accuracy: 0.5250 - val_loss: 1.1726 - val_accuracy: 0.5956
Epoch 7/10
5022/5022 [==============================] - 737s

## 5. Evaluation

In [ ]:
# ── Results comparison table ──────────────────────────────────────────────────
PREV_ACC = 0.6457
PREV_F1  = 0.5824

print(f"\n{'─'*76}")
print(f"{'Model':<40} {'Test Acc':>10} {'Test F1':>10} {'Δ Acc':>7} {'Δ F1':>7}")
print(f"{'─'*76}")
print(f"{'DistilBERT OPTIMAL v1 (baseline)':<40} {PREV_ACC:>10.4f} {PREV_F1:>10.4f} {'—':>7} {'—':>7}")
for (mname, gamma), r in all_results.items():
    p2_ep = len(r['hist_p2'].history['accuracy'])
    label = f"{mname}  γ={gamma}  ({p2_ep} ep P2)"
    print(f"{label:<40} {r['test_acc']:>10.4f} {r['test_f1']:>10.4f}"
          f" {r['test_acc']-PREV_ACC:>+7.4f} {r['test_f1']-PREV_F1:>+7.4f}")
print(f"{'─'*76}")


In [ ]:
# ── Training curves — one row per experiment ──────────────────────────────────
n_exp = len(all_results)
fig, axes = plt.subplots(n_exp, 2, figsize=(14, 5 * n_exp))
if n_exp == 1:
    axes = axes[np.newaxis, :]

for row, ((mname, gamma), r) in enumerate(all_results.items()):
    h1   = r["hist_p1"].history
    h2   = r["hist_p2"].history
    p1_n = len(h1["accuracy"])

    all_acc     = h1["accuracy"]     + h2["accuracy"]
    all_val_acc = h1["val_accuracy"] + h2["val_accuracy"]
    all_loss    = h1["loss"]         + h2["loss"]
    all_val_lss = h1["val_loss"]     + h2["val_loss"]
    epochs = range(1, p1_n + len(h2["accuracy"]) + 1)

    for col, (train_v, val_v, title) in enumerate([
        (all_acc,  all_val_acc, "Accuracy"),
        (all_loss, all_val_lss, "Loss"),
    ]):
        ax = axes[row, col]
        ax.plot(epochs, train_v, label="Train")
        ax.plot(epochs, val_v,   label="Validation")
        ax.axvline(p1_n + 0.5, color="gray", linestyle="--",
                   alpha=0.6, label="Phase 1→2")
        ax.set_title(f"{mname}  γ={gamma}  {title}")
        ax.set_xlabel("Epoch")
        ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("RoBERTa Fine-tuning — 34 Consolidated Classes",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("training_curves_comparison.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
# ── Per-class F1: side-by-side comparison ────────────────────────────────────
all_class_f1 = {}
for (mname, gamma), r in all_results.items():
    rep = classification_report(
        test_labels, r["test_preds"],
        labels=np.arange(NUM_CLASSES), target_names=label_names,
        output_dict=True, zero_division=0,
    )
    all_class_f1[(mname, gamma)] = {cls: rep[cls]["f1-score"] for cls in label_names}

ref_key    = list(all_results.keys())[0]
sorted_cls = sorted(label_names, key=lambda c: all_class_f1[ref_key][c])

fig, ax = plt.subplots(figsize=(15, 8))
x       = np.arange(NUM_CLASSES)
bar_w   = 0.8 / max(len(all_results), 1)
palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

for i, ((mname, gamma), f1_dict) in enumerate(all_class_f1.items()):
    scores = [f1_dict[cls] for cls in sorted_cls]
    label  = f"{mname} γ={gamma}"
    ax.barh(x + i * bar_w, scores, bar_w,
            label=label, color=palette[i], alpha=0.85)

ax.set_yticks(x + bar_w / 2)
ax.set_yticklabels(sorted_cls, fontsize=8)
ax.axvline(0.50, color="red",   linestyle="--", alpha=0.4, label="F1=0.50")
ax.axvline(0.75, color="green", linestyle="--", alpha=0.4, label="F1=0.75")
ax.set_xlabel("F1 Score")
ax.set_title(f"Per-Class F1 — RoBERTa comparison ({NUM_CLASSES} classes)")
ax.legend(); plt.tight_layout()
plt.savefig("per_class_f1_comparison.png", dpi=100)
plt.show()

for (mname, gamma), f1_dict in all_class_f1.items():
    sf = sorted(f1_dict.items(), key=lambda x: x[1])
    print(f"\n{mname} γ={gamma}  — 5 hardest:")
    for cls, sc in sf[:5]:  print(f"  {cls:<28}: {sc:.3f}")
    print(f"{mname} γ={gamma}  — 5 easiest:")
    for cls, sc in sf[-5:]: print(f"  {cls:<28}: {sc:.3f}")


In [ ]:
# ── Sample misclassifications — best experiment ───────────────────────────────
best_key = max(all_results, key=lambda k: all_results[k]["test_acc"])
r        = all_results[best_key]
mname, gamma = best_key
print(f"Misclassifications from best experiment "
      f"({mname} γ={gamma}, test_acc={r['test_acc']:.4f}):\n")

wrong_idx = np.where(r["test_preds"] != test_labels)[0][:10]
for idx in wrong_idx:
    true = label_names[test_labels[idx]]
    pred = label_names[r["test_preds"][idx]]
    text = test_texts[idx][:110]
    print(f"  True: {true:<22} → Pred: {pred}")
    print(f"  Text: {text}...")
    print()
